# 03c: Annotation and results (interactive)

**Step 3 of 3 of the analysis of one sample.** 03a and 03b have already run for the sample (through `run_batch.py`), so the clusters, UMAPs, dotplots, matrixplots, cluster summary and cluster maps are ready in `<sample>/figures/`. This notebook loads the clustered data in seconds; no segmentation or clustering is repeated.

What you do here:
1. look at the clusters (4.5) and, where needed, split mixed clusters (4.6, `SUBCLUSTER`);
2. name the clusters (4.7, `cluster_to_ct_dict`);
3. check the annotation (4.10) and run Part 5 (results) and Part 6 (save).

Tip: save a copy of this notebook per sample (e.g. `ss1_03c_annotate.ipynb` in the sample folder), so each sample keeps its own `SUBCLUSTER` and `cluster_to_ct_dict`.

| Part | Step |
|---|---|
| 4 | Subclustering, annotation, QC |
| 5 | Composition, immune context, Ki67/HIF1A, density maps, neighbourhoods, distances |
| 6 | Save the annotated data and the one-row sample summary |

All figures go to the same `<sample>/figures/` folder as those of 03a and 03b.

**Figures for the thesis.** Every figure is drawn at its final printed size: full text width (`FIG_FULL`, 6.3 in) or, for the UMAPs, about two thirds of it, with the same font (Arial, 8 pt; panel titles 9 pt) and line widths, and saved at 300 dpi. Insert the PNG files into Word at **100 %** (Word reads the size from the file), then all figures have the same font size on the page. The sample label is shown above each figure on screen but not written into the files, because the figure caption carries it (`FIGURE_TITLES = True` to keep it). Add `'pdf'` to `FIG_FORMATS` for vector files.


---
# Part 0: Setup
## 0.1 Sample

In [ ]:
SAMPLE      = 'ss1'                                # <-- EDIT
OUTPUT_ROOT = 'path/to/output'      # <-- EDIT same as in run_batch.py

## 0.2 Load the clustered data, settings and helpers

In [ ]:
import os, json, warnings, logging
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.spatial import cKDTree
from IPython.display import display, Image

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'              # silence TensorFlow messages from importing SPACEc
logging.getLogger('absl').setLevel(logging.ERROR)
import spacec as sp
import scanpy as sc
# ── figure style for the thesis: every figure is drawn at its final printed size ──────────────
FIG_FULL      = 6.3        # <-- EDIT inches: text width of the thesis page (A4 or Letter with 2.5 cm margins)
FIG_HALF      = 3.1        #     half width (two figures side by side)
FONT_SIZE     = 8          # pt in print: axis labels, tick labels, legends
TITLE_SIZE    = 9          # pt in print: panel titles
SAVE_DPI      = 300        # resolution of the saved files
FIG_FORMATS   = ('png',)   # <-- EDIT add 'pdf' for vector files, e.g. ('png', 'pdf')
FIGURE_TITLES = False      # <-- EDIT True: '<sample> | <title>' also on the saved files (it is always shown on screen)
SCREEN_DPI    = 80         # the 03b figures shown in 4.5 (same size as in 03b)
DISPLAY_DPI   = 110        # on-screen size of the figures made here (8 pt stays readable)

def fig_style():
    """One font, one size and thin lines for all figures; call again after anything that resets matplotlib."""
    sc.settings.set_figure_params(dpi=DISPLAY_DPI, dpi_save=SAVE_DPI, fontsize=FONT_SIZE, facecolor='white')
    plt.rcParams.update({
        'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'font.size': FONT_SIZE, 'axes.titlesize': TITLE_SIZE, 'axes.titleweight': 'normal',
        'axes.labelsize': FONT_SIZE, 'xtick.labelsize': FONT_SIZE, 'ytick.labelsize': FONT_SIZE,
        'legend.fontsize': FONT_SIZE, 'legend.title_fontsize': FONT_SIZE, 'figure.titlesize': TITLE_SIZE,
        'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
        'xtick.major.size': 2.5, 'ytick.major.size': 2.5, 'lines.linewidth': 1.0,
        'axes.grid': False, 'savefig.dpi': SAVE_DPI, 'pdf.fonttype': 42, 'svg.fonttype': 'none'})
fig_style()

def grid_figsize(n_rows):
    """Size (in) of a dotplot / matrixplot with n_rows groups at full text width."""
    return (FIG_FULL, 0.2 * n_rows + 1.5)

output_dir = os.path.join(OUTPUT_ROOT, SAMPLE) + os.sep
OUT = output_dir + SAMPLE + '_'
RES = OUT                                               # results go into the sample folder
FIG_DIR = output_dir + 'figures/'                       # one figure folder per sample
os.makedirs(FIG_DIR, exist_ok=True)

cfg = json.load(open(OUT + 'run_config.json'))           # from 03a
globals().update({k: cfg[k] for k in ('CONDITION', 'REGION', 'PLOT_TAG', 'H', 'W', 'BG_DS', 'NUCLEUS_CHANNEL',
                                      'marker_channels', 'LINEAGE_MARKERS', 'STATE_MARKERS', 'NUCLEAR_MARKERS',
                                      'NONFUNCTIONAL', 'REF_CHANNELS')})
PHENO_MARKERS = LINEAGE_MARKERS + STATE_MARKERS
ccfg = json.load(open(OUT + 'cluster_config.json'))      # from 03b
globals().update({k: ccfg[k] for k in ('REGION_GEOJSON', 'ROI', 'ROI_AREA_MM2', 'PIXEL_SIZE_UM', 'CLUSTER_KEY',
                                       'CLUSTER_MARKERS', 'PLOT_MARKERS', 'clustering_random_seed', 'WEAK_Z')})
PARAM_LOG = json.load(open(OUT + 'parameters.json'))
adata = sc.read_h5ad(OUT + 'adata_clustered.h5ad')
adata.obs[CLUSTER_KEY] = adata.obs[CLUSTER_KEY].astype(str).astype('category')

def finish(name, title=None, fig=None, dpi=SAVE_DPI, tight=True):
    """Save the figure as figures/<PLOT_TAG>_<name>.<png/pdf> and show it with '<PLOT_TAG> | <title>' above it.
    The saved file has no title unless FIGURE_TITLES = True (in the thesis the caption says what it shows)."""
    fig = fig or plt.gcf()
    label = PLOT_TAG + (f'  |  {title}' if title else '')
    def add_title():
        if fig.get_constrained_layout():
            fig.suptitle(label, fontsize=TITLE_SIZE, fontweight='bold')
            return
        # place the title above everything already drawn (panel titles, legends), so nothing overlaps
        top = fig.get_tightbbox(fig.canvas.get_renderer()).y1 / fig.get_figheight()
        fig.suptitle(label, fontsize=TITLE_SIZE, fontweight='bold', y=max(top, 1.0) + 0.01, va='bottom')
    if FIGURE_TITLES:
        add_title()
    for ext in FIG_FORMATS:
        fig.savefig(FIG_DIR + f'{PLOT_TAG}_{name}.{ext}', dpi=dpi,
                    **(dict(bbox_inches='tight', pad_inches=0.04) if tight else {}))
    if not FIGURE_TITLES:
        add_title()                                     # on screen only
    fig.set_dpi(DISPLAY_DPI)
    plt.show()

def cluster_summary(adata, key, markers=PLOT_MARKERS, thr=1.0):
    """Size, high markers (mean z > thr) and a plausibility flag per cluster."""
    X = adata.to_df()[markers]; cl = adata.obs[key]
    means = X.groupby(cl, observed=True).mean()
    n = cl.value_counts().reindex(means.index)
    high = means[CLUSTER_MARKERS].apply(lambda r: ', '.join(f'{m} {v:.1f}' for m, v in r[r > thr].sort_values(ascending=False).items()) or '-', axis=1)
    state = means[[m for m in STATE_MARKERS if m in means]].apply(lambda r: ', '.join(f'{m} {v:.1f}' for m, v in r[r > thr].items()) or '-', axis=1)
    n_lin = (means[CLUSTER_MARKERS] > 0.5).sum(axis=1)
    top = means[CLUSTER_MARKERS].max(axis=1)
    flag = np.select([n_lin == 0, top < WEAK_Z, n_lin >= 4],
                     ['no lineage marker -> tumour by exclusion?',
                      f'weak (max z < {WEAK_Z:g}) -> check map',
                      'many markers -> doublet/spillover?'], '')
    return pd.DataFrame({'n_cells': n, 'pct': (100 * n / n.sum()).round(1), f'lineage z>{thr}': high,
                         f'state z>{thr}': state, 'flag': flag})

def cluster_maps(adata, key, name, title, ncol=6):
    """One small tissue map per cluster / cell type (red = cells of that group)."""
    cats = list(adata.obs[key].cat.categories)
    xy = adata.obs[['x', 'y']].to_numpy()
    nr = int(np.ceil(len(cats) / ncol))
    pw = FIG_FULL / ncol                                    # panel width (in) at full text width
    fig, axes = plt.subplots(nr, ncol, figsize=(FIG_FULL, nr * (pw * H / W + 0.25)), squeeze=False)
    for ax, c in zip(axes.ravel(), cats):
        m = (adata.obs[key] == c).to_numpy()
        ax.scatter(xy[:, 0], xy[:, 1], s=0.02, c='0.88', rasterized=True, linewidths=0)
        ax.scatter(xy[m, 0], xy[m, 1], s=0.12, c='crimson', rasterized=True, linewidths=0)
        ax.set_title(f'{c} (n={m.sum():,})', fontsize=FONT_SIZE - 1, pad=2)
        ax.invert_yaxis(); ax.set_aspect('equal'); ax.axis('off')
    for ax in axes.ravel()[len(cats):]:
        ax.axis('off')
    plt.tight_layout(pad=0.3, w_pad=0.3, h_pad=0.6); finish(name, title)


print(f'{PLOT_TAG}: {adata.n_obs:,} cells, {adata.obs[CLUSTER_KEY].nunique()} clusters, region {ROI_AREA_MM2:.2f} mm²')

---
# Part 4: Annotation
## 4.5 Clusters from 03b
The figures made by 03b, shown again here: UMAP, marker UMAPs, dotplot, matrixplot and cluster maps, followed by the cluster summary (size, markers with mean z > 1, flags). Use them to decide which clusters to split (4.6) and how to name them (4.7).

In [ ]:
from PIL import Image as PILImage
for name in ['4-05_umap_clusters', '4-05_umap_markers', '4-05_dotplot_clusters', '4-05_matrixplot_clusters',
             '4-05_cluster_maps']:
    p = FIG_DIR + f'{PLOT_TAG}_{name}.png'
    if os.path.exists(p):
        with PILImage.open(p) as im:               # figures are saved at 200-300 dpi: scale back to screen size
            width = int(im.width * SCREEN_DPI / im.info.get('dpi', (300, 300))[0])
        display(Image(filename=p, width=width))
pd.set_option('display.max_colwidth', None)
cluster_summary(adata, CLUSTER_KEY)

## 4.6 Subclustering
Clusters that mix several cell types, for example a vessel cluster containing both endothelial and mural cells, are split again with Leiden at their own resolution (`sc.tl.leiden` with `restrict_to`). Subclusters are named `<cluster>,0`, `<cluster>,1`, …, and all other clusters keep their ID.

In [ ]:
MERGE      = {}               # <-- EDIT clusters merged into one before subclustering, e.g. {'6': '6+8', '8': '6+8'}; {} = none
SUBCLUSTER = {}               # <-- EDIT {cluster ID: resolution}, e.g. {'10': 0.3, '6+8': 0.3}; {} = no subclustering
                              #     a subcluster can be split again by listing it after its parent, e.g. '12,2': 0.3
SUB_KEY = CLUSTER_KEY + '_subcluster'

adata.obs[SUB_KEY] = adata.obs[CLUSTER_KEY].astype(str).replace(MERGE).astype('category')   # must be categorical
PARAM_LOG['merge_before_subclustering'] = MERGE
for c, res in SUBCLUSTER.items():
    if str(c) not in adata.obs[SUB_KEY].cat.categories:
        print(f'cluster {c} not found - skipped'); continue
    sc.tl.leiden(adata, seed=clustering_random_seed, restrict_to=(SUB_KEY, [str(c)]),
                 resolution=res, key_added=SUB_KEY)
    adata.obs[SUB_KEY] = adata.obs[SUB_KEY].astype('category')
PARAM_LOG['subclustering'] = SUBCLUSTER
print(adata.obs[SUB_KEY].nunique(), 'clusters after subclustering')
if SUBCLUSTER:
    n_sub = adata.obs[SUB_KEY].nunique()
    sc.pl.dotplot(adata, PLOT_MARKERS, SUB_KEY, dendrogram=False, figsize=grid_figsize(n_sub), show=False)
    finish('4-06_dotplot_subclusters', 'Marker expression after subclustering')

    # matrixplot: mean z-score per cluster, all clusters
    sc.pl.matrixplot(adata, PLOT_MARKERS, SUB_KEY, dendrogram=False, colorbar_title='mean z-score',
                     vmin=-2, vmax=2, cmap='RdBu_r', figsize=grid_figsize(n_sub), show=False)
    finish('4-06_matrixplot_subclusters', 'Mean z-score per cluster after subclustering')

    # matrixplot: only the clusters that were split, for a closer look
    is_sub = adata.obs[SUB_KEY].astype(str).str.contains(',')
    ad_sub = adata[is_sub].copy()
    ad_sub.obs[SUB_KEY] = ad_sub.obs[SUB_KEY].cat.remove_unused_categories()
    sc.pl.matrixplot(ad_sub, PLOT_MARKERS, SUB_KEY, dendrogram=False, colorbar_title='mean z-score',
                     vmin=-2, vmax=2, cmap='RdBu_r', figsize=grid_figsize(ad_sub.obs[SUB_KEY].nunique()), show=False)
    finish('4-06_matrixplot_subclusters_only', 'Mean z-score of the subclusters')

    display(cluster_summary(adata, SUB_KEY))

## 4.7 Cell-type annotation
Each (sub)cluster is assigned a cell type from its marker profile (4.5–4.6) and its location in the tissue. Clusters with the same name are merged.

| Cell type | Defining markers |
|---|---|
| Tumour | no lineage marker (synovial sarcoma has no specific marker in this panel; tumour by exclusion) |
| CD8 T cell / CD4-like T cell | CD45, CD3E with / without CD8 (CD4 antibody not functional) |
| Macrophage | CD68, CD45 |
| Other immune | CD45 only |
| Immune (mixed) | several immune lineages at once (doublets / dense contact) |
| Endothelial | CD31, VWF, CD34 |
| Mural cell / Pericyte | aSMA (+ PDGFRb) / PDGFRb (low aSMA) |
| Fibroblast | col1A1 |
| CD34⁺ stromal cell | CD34 without CD31 |
| PanCK⁺ cell | PanCK |

Clusters named `'Noise'` are removed. Clusters named `'recluster'` are split again in 4.8. `print_dict_template` lists every cluster with its markers as a starting point for the dictionary.

In [ ]:
def print_dict_template(adata, key):
    """Print an empty annotation dictionary with each cluster's markers as a comment."""
    s = cluster_summary(adata, key)
    print('cluster_to_ct_dict = {')
    for c, r in s.iterrows():
        hint = r[[k for k in s.columns if k.startswith('lineage')][0]]
        print(f"    {str(c)!r:>10}: '',   # n={r.n_cells:,}  {hint}  {r.flag}")
    print('}')

print_dict_template(adata, SUB_KEY)

In [ ]:
cluster_to_ct_dict = {      # <-- EDIT sample-specific: cluster ID -> cell type (copy the template printed above)
}
assert cluster_to_ct_dict, 'fill in cluster_to_ct_dict (copy the template printed by the cell above)'

def apply_annotation(adata, src_key, mapping, new_key):
    """Map cluster IDs to cell-type names; unnamed clusters become 'unannotated'."""
    missing = sorted(set(adata.obs[src_key].astype(str)) - set(mapping))
    if missing:
        print('WARNING: clusters without a name (set to "unannotated"):', missing)
    adata.obs[new_key] = adata.obs[src_key].astype(str).map(mapping).fillna('unannotated').astype('category')

apply_annotation(adata, SUB_KEY, cluster_to_ct_dict, 'cell_type_coarse')
adata = adata[~adata.obs['cell_type_coarse'].isin(['Noise'])].copy()
adata.obs['cell_type'] = adata.obs['cell_type_coarse']
PARAM_LOG['annotation'] = cluster_to_ct_dict
print(adata.obs['cell_type'].value_counts().to_string())

## 4.8 Second annotation round (only if clusters were named `'recluster'`)
Cells named `'recluster'` are clustered again at `RECLUSTER_RES` and named in `cluster_to_ct_dict_2`. Both cells skip themselves when nothing is named `'recluster'`.

In [ ]:
RECLUSTER_RES = 1.0        # <-- EDIT

if 'recluster' in set(adata.obs['cell_type_coarse']):
    sc.tl.leiden(adata, seed=clustering_random_seed, restrict_to=('cell_type_coarse', ['recluster']),
                 resolution=RECLUSTER_RES, key_added='cell_type_coarse_subcluster')
    sc.pl.dotplot(adata, PLOT_MARKERS, 'cell_type_coarse_subcluster', dendrogram=False,
                  figsize=grid_figsize(adata.obs['cell_type_coarse_subcluster'].nunique()), show=False)
    finish('4-08_dotplot_recluster', 'Second annotation round')
    print_dict_template(adata[adata.obs['cell_type_coarse'] == 'recluster'], 'cell_type_coarse_subcluster')
else:
    print('nothing named "recluster" - skipped')

In [ ]:
if 'recluster' in set(adata.obs['cell_type_coarse']):
    cluster_to_ct_dict_2 = {t: t for t in adata.obs['cell_type_coarse'].cat.categories if t != 'recluster'}
    cluster_to_ct_dict_2.update({   # <-- EDIT names for 'recluster,0', 'recluster,1', ...
    })
    apply_annotation(adata, 'cell_type_coarse_subcluster', cluster_to_ct_dict_2, 'cell_type')
    adata = adata[~adata.obs['cell_type'].isin(['Noise'])].copy()
    PARAM_LOG['annotation_round2'] = cluster_to_ct_dict_2
    print(adata.obs['cell_type'].value_counts().to_string())

## 4.9 Manual exclusion of artefact regions (optional)
Staining artefacts that were not caught automatically (for example a dense, non-specific signal in one channel at a tissue edge) can be excluded with rectangular regions. All cells inside a region are removed. Leave `EXCLUDE_BOXES` empty to skip this step. Report any exclusion in the Methods; the number of removed cells is stored in the parameter log.

In [ ]:
EXCLUDE_BOXES = []          # <-- EDIT list of (x_min, x_max, y_min, y_max) in px; [] = none

if EXCLUDE_BOXES:
    inside = np.zeros(adata.n_obs, bool)
    for bx0, bx1, by0, by1 in EXCLUDE_BOXES:
        inside |= (adata.obs.x.between(bx0, bx1) & adata.obs.y.between(by0, by1)).to_numpy()
    print(f'{inside.sum():,} cells inside the excluded regions:')
    print(adata.obs['cell_type'][inside].value_counts().to_string())
    fig, ax = plt.subplots(figsize=(FIG_FULL, FIG_FULL * H / W))
    ax.scatter(adata.obs.x, adata.obs.y, s=0.05, c='0.85', rasterized=True, linewidths=0)
    ax.scatter(adata.obs.x[inside], adata.obs.y[inside], s=0.4, c='red', rasterized=True, linewidths=0)
    for bx0, bx1, by0, by1 in EXCLUDE_BOXES:
        ax.add_patch(plt.Rectangle((bx0, by0), bx1 - bx0, by1 - by0, fill=False, ec='black', lw=1))
    ax.invert_yaxis(); ax.set_aspect('equal'); ax.axis('off'); ax.set_title('excluded cells (red)')
    finish('4-09_excluded_regions', 'Manually excluded artefact regions')
    adata = adata[~inside].copy()
    PARAM_LOG['manual_exclusion'] = dict(boxes=EXCLUDE_BOXES, n_cells=int(inside.sum()))
else:
    print('no regions excluded')

## 4.10 Annotation quality control
* **Dotplot / matrixplot per cell type**: each type should be high in its defining markers (table in 4.7), and tumour should be low in all lineage markers.
* **Cell-type map**: tumour in grey, all other cell types drawn on top in contrasting colours.
* **Map per cell type**: vessels should appear as lines and rings, immune cells as infiltrates or aggregates, stroma as bands.

In [ ]:
n_ct = adata.obs['cell_type'].nunique()
sc.pl.dotplot(adata, PLOT_MARKERS, 'cell_type', dendrogram=False, figsize=grid_figsize(n_ct), show=False)
finish('4-10_dotplot_cell_types', 'Marker expression per cell type')

sc.pl.matrixplot(adata, PLOT_MARKERS, 'cell_type', dendrogram=False, colorbar_title='mean z-score',
                 vmin=-2, vmax=2, cmap='RdBu_r', figsize=grid_figsize(n_ct), show=False)
finish('4-10_matrixplot_cell_types', 'Mean z-score per cell type')
print(adata.obs['cell_type'].value_counts().to_string())

In [ ]:
# one colour per cell type, the same for all samples (bright and contrasting; tumour grey)
CELL_TYPE_COLORS = {                 # <-- EDIT add new cell-type names here
    'Tumour':             '#d9d9d9',   # light grey
    'CAF':                '#ff7f00',   # orange
    'ECM':                '#fdbf6f',   # light orange
    'Endothelial':        '#e41a1c',   # red
    'Endothelial (weak)': '#fb9a99',   # light red
    'Pericyte':           '#f781bf',   # pink
    'Macrophage':         '#4daf4a',   # green
    'CD14+ myeloid':      '#b2df8a',   # light green
    'Monocyte':           '#b2df8a',   # light green
    'CD8 T cell':         '#0000ff',   # blue
    'CD4-like T cell':    '#00bfff',   # light blue
    'B cell':             '#6a3d9a',   # purple
    'Epithelial':         '#ff00ff',   # magenta
    'VEGFR1+ / CD31-':    '#7cfc00',   # lawn green
    'TIE2+ / CD31-':      '#808000',   # olive
    'CD206+ / CD68-':     '#a65628',   # brown
    'VWF+ / CD206+':      '#8dd3c7',   # teal
    'Unassigned':         '#ffff33',   # yellow
    'Mix':                '#000000',   # black
}
present = list(adata.obs['cell_type'].cat.categories)
for tname, colr in zip([x for x in present if x not in CELL_TYPE_COLORS],
                       ['#00ff7f', '#8b0000', '#008080', '#ff1493', '#4b0082']):
    CELL_TYPE_COLORS[tname] = colr                           # fallback colours for new names
    print(f'no colour for {tname!r} - using {colr}; add it to CELL_TYPE_COLORS')
CELL_TYPE_COLORS = {k: v for k, v in CELL_TYPE_COLORS.items() if k in present}   # only the types of this sample
adata.uns['cell_type_colors'] = [CELL_TYPE_COLORS[x] for x in present]
# cell-type map: tumour first and small, all other types on top and larger
DOT_TUMOUR, DOT_OTHER = 0.15, 0.8  # <-- EDIT dot sizes (pt²) in the printed figure
LEGEND_W = 1.7                     # inches reserved for the legend; map + legend = full text width
map_w = FIG_FULL - LEGEND_W
fig, ax = plt.subplots(figsize=(map_w, map_w * H / W))
handles = []
for t in ['Tumour'] + [t for t in CELL_TYPE_COLORS if t != 'Tumour']:
    m = (adata.obs['cell_type'] == t).to_numpy()
    if not m.any():
        continue
    ax.scatter(adata.obs.x[m], adata.obs.y[m], s=DOT_TUMOUR if t == 'Tumour' else DOT_OTHER,
               c=CELL_TYPE_COLORS[t], rasterized=True, linewidths=0)
    handles.append(Line2D([0], [0], marker='o', linestyle='', markersize=5, markerfacecolor=CELL_TYPE_COLORS[t],
                          markeredgecolor='0.3', markeredgewidth=0.4, label=f'{t} (n={m.sum():,})'))
ax.invert_yaxis(); ax.set_aspect('equal'); ax.axis('off')
ax.legend(handles=handles, loc='center left', bbox_to_anchor=(1.0, 0.5), frameon=False,
          handletextpad=0.3, labelspacing=0.5)
finish('4-10_cell_type_map', 'Cell-type map')

cluster_maps(adata, 'cell_type', '4-10_cell_type_maps', 'Spatial location of each cell type', ncol=4)


### UMAP of the final annotation
The UMAP from 03b (same embedding), coloured by the final clusters after subclustering and by the cell types. Each cell type should form one or a few coherent regions; a cell type scattered across the whole UMAP, or a cluster split between two cell types, points to an annotation that needs another look.

In [ ]:
# UMAP of the final annotation: the 03b embedding, coloured by final (sub)cluster and by cell type
UMAP_SIZE = 3.0            # inches (square panel); with the legend the figure is about 2/3 of the text width
if 'X_umap' in adata.obsm:
    adata.obs[SUB_KEY] = adata.obs[SUB_KEY].astype('category').cat.remove_unused_categories()   # removed (Noise) clusters
    dot = max(0.5, min(5, 60000 / adata.n_obs))                                                  # point size for this many cells
    with plt.rc_context({'figure.figsize': (UMAP_SIZE, UMAP_SIZE)}):
        sc.pl.umap(adata, color=SUB_KEY, legend_loc='on data', legend_fontsize=FONT_SIZE - 2,
                   legend_fontoutline=1.5, size=dot, title='', frameon=True, show=False)
        finish('4-10_umap_final_clusters', 'UMAP of the final clusters')
        sc.pl.umap(adata, color='cell_type', palette=CELL_TYPE_COLORS, size=dot, title='', frameon=True, show=False)
        finish('4-10_umap_cell_types', 'UMAP of the cell types')
else:
    print('no UMAP in the data (X_umap missing): run 03b again for this sample')

---
# Part 5: Results
## 5.1 Cell-type composition table
Number and percentage of cells per cell type. `PIXEL_SIZE_UM` converts distances and areas to µm and mm². `STATE_Z` is the z-score above which a cell counts as Ki67⁺ or HIF1A⁺. The cell types are also grouped into tissue compartments: tumour, stroma, vascular, immune, epithelial.

In [ ]:
STATE_Z       = 1.5      # <-- EDIT z-score threshold for Ki67+ / HIF1A+

TISSUE_COMPARTMENT = {               # <-- EDIT if cell types are added or renamed; unlisted names -> 'Other'
    'Tumour': 'Tumour',
    'CAF': 'Stroma', 'ECM': 'Stroma', 'Fibroblast': 'Stroma', 'CD34+ stromal cell': 'Stroma',
    'Endothelial': 'Vascular', 'Endothelial (weak)': 'Vascular', 'Pericyte': 'Vascular', 'Mural cell': 'Vascular',
    'Macrophage': 'Immune', 'CD14+ myeloid': 'Immune', 'Monocyte': 'Immune',
    'CD8 T cell': 'Immune', 'CD4-like T cell': 'Immune', 'B cell': 'Immune',
    'Other immune': 'Immune', 'Immune (mixed)': 'Immune',
    'Epithelial': 'Epithelial', 'PanCK+ cell': 'Epithelial',
    # 'Other': Mix, Unassigned, VEGFR1+ / CD31-, TIE2+ / CD31-, CD206+ / CD68-, VWF+ / CD206+
}
COMP_COLORS = {'Tumour': '#bdbdbd', 'Stroma': '#ff7f00', 'Vascular': '#e41a1c',
               'Immune': '#1f4fff', 'Epithelial': '#ff00ff', 'Other': '#666666'}
IMMUNE_TYPES = [t for t, c in TISSUE_COMPARTMENT.items() if c == 'Immune']

obs = adata.obs.copy()
obs['cell_type'] = obs['cell_type'].astype(str)
obs['compartment'] = obs['cell_type'].map(TISSUE_COMPARTMENT).fillna('Other')
print('counted as Other:', sorted(set(obs['cell_type']) - set(TISSUE_COMPARTMENT)) or '-')
counts = obs['cell_type'].value_counts()
pct = 100 * counts / counts.sum()
col = lambda t: CELL_TYPE_COLORS.get(t, '#999999')

summary_tab = pd.DataFrame({'n_cells': counts, 'percent': pct.round(2)})
summary_tab.insert(0, 'condition', CONDITION); summary_tab.insert(0, 'sample', SAMPLE)
summary_tab.to_csv(FIG_DIR + f'{PLOT_TAG}_5-01_cell_type_percentages.csv')
summary_tab

## 5.2 Cell-type composition: bar plot
Percentage of all cells per cell type, with the absolute number of cells.

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_FULL, 0.22 * len(counts) + 0.6))
order = pct.sort_values().index
bars = ax.barh(order, pct[order], color=[col(t) for t in order], edgecolor='0.3', linewidth=0.4, height=0.7)
for b, t in zip(bars, order):
    ax.text(b.get_width() + pct.max() * 0.01, b.get_y() + b.get_height() / 2,
            f'{pct[t]:.1f} %  (n = {counts[t]:,})', va='center', fontsize=FONT_SIZE - 1)
ax.set_xlabel('% of all cells'); ax.set_xlim(0, pct.max() * 1.3)
ax.set_title(f'n = {counts.sum():,} cells')
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.tight_layout(); finish('5-02_bar_cell_types', 'Cell-type composition')

## 5.3 Cell-type composition: donut charts
Left: all cells. Right: the tumour microenvironment only, with tumour cells excluded, so the smaller populations become visible.

In [ ]:
def donut(ax, series, title, min_label_pct=3.0):
    """Donut chart with % labels on wedges >= min_label_pct and a legend with all values below it."""
    s = series.sort_values(ascending=False)
    p = 100 * s / s.sum()
    wedges, _ = ax.pie(s, colors=[col(t) for t in s.index], startangle=90, counterclock=False,
                       wedgeprops=dict(width=0.42, edgecolor='white', linewidth=0.8))
    for w, t in zip(wedges, s.index):
        if p[t] >= min_label_pct:
            ang = np.deg2rad((w.theta1 + w.theta2) / 2)
            ax.text(0.79 * np.cos(ang), 0.79 * np.sin(ang), f'{p[t]:.0f}%', ha='center', va='center',
                    fontsize=FONT_SIZE - 2, fontweight='bold', color='0.2' if t == 'Tumour' else 'white')
    ax.legend(wedges, [f'{t}  {p[t]:.1f} %' for t in s.index], loc='upper center', bbox_to_anchor=(0.5, -0.02),
              ncol=2, frameon=False, fontsize=FONT_SIZE - 1, handlelength=1, columnspacing=1, labelspacing=0.3)
    ax.text(0, 0, f'n = {s.sum():,}', ha='center', va='center', fontsize=FONT_SIZE - 1)
    ax.set_title(title, pad=6)

fig, axes = plt.subplots(1, 2, figsize=(FIG_FULL, 3.6))
donut(axes[0], counts, 'All cells')
donut(axes[1], counts.drop('Tumour', errors='ignore'), 'Tumour microenvironment\n(tumour cells excluded)')
plt.tight_layout(w_pad=1.0); finish('5-03_pie_cell_types', 'Cell-type composition')

## 5.4 Tissue compartments, immune composition and key numbers
* **Left**: the share of each tissue compartment.
* **Right**: the composition of the immune infiltrate.

The table gives immune ratios and densities (cells per mm² of tissue). These are the numbers to compare between samples.

In [ ]:
comp = obs['compartment'].value_counts()
comp_pct = 100 * comp / comp.sum()
imm = obs.loc[obs['cell_type'].isin(IMMUNE_TYPES), 'cell_type'].value_counts()
imm_pct = 100 * imm / max(imm.sum(), 1)

fig, axes = plt.subplots(1, 2, figsize=(FIG_FULL, 2.3), gridspec_kw={'width_ratios': [1.25, 1]})
left = 0
for c in comp_pct.index:                                     # stacked compartment bar
    axes[0].barh(0, comp_pct[c], left=left, color=COMP_COLORS.get(c, '#666'), edgecolor='white', linewidth=0.6)
    if comp_pct[c] > 15:                                     # smaller ones only in the legend
        axes[0].text(left + comp_pct[c] / 2, 0, f'{c}\n{comp_pct[c]:.1f} %', ha='center', va='center',
                     fontsize=FONT_SIZE - 1, color='black' if c == 'Tumour' else 'white', fontweight='bold')
    left += comp_pct[c]
axes[0].set_xlim(0, 100); axes[0].set_yticks([]); axes[0].set_xlabel('% of all cells')
axes[0].set_title('Tissue compartments')
axes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=COMP_COLORS.get(c, '#666')) for c in comp_pct.index],
               labels=[f'{c} ({comp_pct[c]:.1f} %)' for c in comp_pct.index], fontsize=FONT_SIZE - 1,
               loc='upper center', bbox_to_anchor=(0.5, -0.35), ncol=3, frameon=False, handlelength=1)
for s_ in ('top', 'right', 'left'):
    axes[0].spines[s_].set_visible(False)
o = imm_pct.sort_values().index                              # immune composition
axes[1].barh(o, imm_pct[o], color=[col(t) for t in o], edgecolor='0.3', linewidth=0.4, height=0.7)
for k, t in enumerate(o):
    axes[1].text(imm_pct[t] + 1, k, f'{imm_pct[t]:.1f} %', va='center', fontsize=FONT_SIZE - 1)
axes[1].set_xlim(0, imm_pct.max() * 1.3 if len(imm_pct) else 1)
axes[1].set_xlabel('% of immune cells'); axes[1].set_title(f'Immune composition (n = {imm.sum():,})')
for s_ in ('top', 'right'):
    axes[1].spines[s_].set_visible(False)
plt.tight_layout(); finish('5-04_compartments_immune', 'Tissue compartments and immune composition')

# key numbers: ratios and densities
n = lambda t: int(counts.get(t, 0))
area_mm2 = ROI_AREA_MM2                 # area of the analysed region (0.3)
key_numbers = {
    'total cells': int(counts.sum()),
    f'{ROI} area (mm²)': round(area_mm2, 2),
    'immune cells, % of all cells': round(100 * imm.sum() / counts.sum(), 2),
    'CD8 : CD4-like T cells': round(n('CD8 T cell') / max(n('CD4-like T cell'), 1), 2),
    'CD8 T cells : macrophages': round(n('CD8 T cell') / max(n('Macrophage'), 1), 2),
    'T cells : macrophages': round((n('CD8 T cell') + n('CD4-like T cell')) / max(n('Macrophage'), 1), 2),
    'immune cells per 1000 tumour cells': round(1000 * imm.sum() / max(n('Tumour'), 1), 1),
    'CD8 T cells per mm²': round(n('CD8 T cell') / area_mm2, 1),
    'CD4-like T cells per mm²': round(n('CD4-like T cell') / area_mm2, 1),
    'macrophages per mm²': round(n('Macrophage') / area_mm2, 1),
    'endothelial cells per mm²': round(n('Endothelial') / area_mm2, 1),
}
key_numbers = pd.Series(key_numbers, name=PLOT_TAG).to_frame()
key_numbers

## 5.5 Proliferation (Ki67) and hypoxia (HIF1A) per cell type
The percentage of cells in each cell type with a z-score above `STATE_Z` for Ki67 (proliferation) and HIF1A (hypoxia).

In [ ]:
X = adata.to_df()
state = pd.DataFrame({f'{m}+ %': 100 * (X[m] > STATE_Z).groupby(obs['cell_type']).mean()
                      for m in ['Ki67', 'HIF1A'] if m in X.columns})
state = state.loc[pct.sort_values(ascending=False).index.intersection(state.index)]

fig, ax = plt.subplots(figsize=(FIG_FULL, 2.6))
xx = np.arange(len(state)); w = 0.4
for i, (c, cc) in enumerate(zip(state.columns, ['#d62728', '#9467bd'])):
    ax.bar(xx + (i - 0.5) * w, state[c], w, label=c.replace(' %', ''), color=cc)
ax.set_xticks(xx); ax.set_xticklabels(state.index, rotation=40, ha='right', rotation_mode='anchor')
ax.set_ylabel(f'% of cells with z > {STATE_Z}'); ax.legend(frameon=False)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.tight_layout(); finish('5-05_ki67_hif1a', 'Proliferation (Ki67) and hypoxia (HIF1A) per cell type')
state.to_csv(FIG_DIR + f'{PLOT_TAG}_5-05_ki67_hif1a.csv')
state.round(1)

## 5.6 Spatial density maps
The number of cells of a type per hexagon, drawn over a faint outline of the whole tissue. This shows hotspots, e.g. where immune cells concentrate.

In [ ]:
DENSITY_TYPES = None          # <-- EDIT list of cell types, or None = the 6 most frequent non-tumour types
if DENSITY_TYPES is None:
    DENSITY_TYPES = [t for t in counts.index if t not in ('Tumour', 'Mix', 'Unassigned')][:6]
DENSITY_TYPES = [t for t in DENSITY_TYPES if t in counts.index]
GRID = 60                                         # hexagons across the image width
nc = 3; nr = int(np.ceil(len(DENSITY_TYPES) / nc))
pw = FIG_FULL / nc
fig, axes = plt.subplots(nr, nc, figsize=(FIG_FULL, nr * (pw * H / W + 0.35)), squeeze=False)
ext = (0, W, 0, H)
for ax, t in zip(axes.ravel(), DENSITY_TYPES):
    m = (obs['cell_type'] == t).to_numpy()
    ax.hexbin(obs.x, obs.y, gridsize=GRID, extent=ext, mincnt=1, cmap='Greys', alpha=0.15)   # tissue outline
    hb = ax.hexbin(obs.x[m], obs.y[m], gridsize=GRID, extent=ext, mincnt=1, cmap='magma_r')
    cb = plt.colorbar(hb, ax=ax, fraction=0.035, pad=0.02, shrink=0.75); cb.ax.tick_params(labelsize=FONT_SIZE - 2)
    cb.set_label('cells / hexagon', fontsize=FONT_SIZE - 2)
    ax.set_title(f'{t} (n={m.sum():,})', fontsize=FONT_SIZE, pad=2)
    ax.invert_yaxis(); ax.set_aspect('equal'); ax.axis('off')
for ax in axes.ravel()[len(DENSITY_TYPES):]:
    ax.axis('off')
plt.tight_layout(pad=0.3); finish('5-06_density_maps', 'Spatial density per cell type')

## 5.7 Neighbourhood enrichment
For every cell, the cell types among its `K` nearest neighbours are counted. This is compared with what would be expected if cell types were randomly distributed, on a log2 scale. **Red** means the two types are neighbours more often than by chance; **blue** means they avoid each other. The diagonal shows how strongly a cell type clusters with itself.

In [ ]:
K = 10      # <-- EDIT number of nearest neighbours (lower = faster)

xy = obs[['x', 'y']].to_numpy()
_, nn = cKDTree(xy).query(xy, k=K + 1)
nn = nn[:, 1:]                                       # drop the cell itself
types = pct.sort_values(ascending=False).index.tolist()
codes = pd.Categorical(obs['cell_type'], categories=types).codes
nb_codes = codes[nn]                                 # neighbour types per cell
global_frac = np.bincount(codes, minlength=len(types)) / len(codes)
enr = np.zeros((len(types), len(types)))
for i in range(len(types)):                          # observed vs expected neighbour frequency
    rows = nb_codes[codes == i].ravel()
    frac = np.bincount(rows, minlength=len(types)) / max(len(rows), 1)
    enr[i] = np.log2((frac + 1e-4) / (global_frac + 1e-4))
enr = pd.DataFrame(enr, index=types, columns=types).replace([np.inf, -np.inf], np.nan).fillna(0.0)
enr.round(2).to_csv(FIG_DIR + f'{PLOT_TAG}_5-07_neighbourhood_enrichment.csv')

side = min(FIG_FULL - 1.2, 0.38 * len(types) + 1.8)        # square heatmap, at most the text width with colorbar
fig, ax = plt.subplots(figsize=(side + 1.2, side), constrained_layout=True)
vals = enr.clip(-3, 3).to_numpy()
im = ax.imshow(vals, cmap='RdBu_r', vmin=-3, vmax=3, aspect='auto')
for i in range(len(types)):
    for j in range(len(types)):
        ax.text(j, i, f'{enr.iat[i, j]:.1f}', ha='center', va='center', fontsize=FONT_SIZE - 2,
                color='white' if abs(vals[i, j]) > 2 else 'black')
ax.set_xticks(np.arange(len(types))); ax.set_xticklabels(types, rotation=45, ha='right')
ax.set_yticks(np.arange(len(types))); ax.set_yticklabels(types)
ax.set_xlabel('neighbour cell type'); ax.set_ylabel('centre cell type')
cb = fig.colorbar(im, ax=ax, ticks=[-3, -2, -1, 0, 1, 2, 3], shrink=0.8)
cb.set_label(f'log2 enrichment among {K} nearest neighbours')
finish('5-07_neighbourhood_enrichment', 'Neighbourhood enrichment', tight=False)

## 5.8 Distances to vessels and to tumour cells
For each immune cell type, the distance (µm) from every cell to the nearest endothelial cell and to the nearest tumour cell.
* Short distances to endothelium mean the cells are perivascular.
* Short distances to tumour cells mean they infiltrate the tumour.

Boxes show the median and interquartile range.

In [ ]:
def dist_to(target, query_mask):
    """Distance (µm) from each query cell to the nearest cell of the target type."""
    tm = (obs['cell_type'] == target).to_numpy()
    if not tm.any():
        return None
    d, _ = cKDTree(xy[tm]).query(xy[query_mask], k=1)
    return d * PIXEL_SIZE_UM

QUERY = [t for t in IMMUNE_TYPES + ['Tumour'] if t in counts.index and t != 'Immune (mixed)']
fig, axes = plt.subplots(1, 2, figsize=(FIG_FULL, 2.8))
dist_summary = {}
for ax, target in zip(axes, ['Endothelial', 'Tumour']):
    data, labels = [], []
    for q in QUERY:
        if q == target:
            continue
        d = dist_to(target, (obs['cell_type'] == q).to_numpy())
        if d is not None and len(d):
            data.append(d); labels.append(q)
            dist_summary[(q, f'median distance to {target.lower()} (µm)')] = round(float(np.median(d)), 1)
    if not data:
        ax.axis('off'); continue
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, widths=0.6,
                    boxprops=dict(linewidth=0.6), whiskerprops=dict(linewidth=0.6),
                    capprops=dict(linewidth=0.6), medianprops=dict(linewidth=0.8))
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=35, ha='right', rotation_mode='anchor')
    for patch, l in zip(bp['boxes'], labels):
        patch.set_facecolor(col(l)); patch.set_alpha(0.85)
    for med in bp['medians']:
        med.set_color('black')
    ax.set_ylabel('distance (µm)')
    ax.set_title(f'Distance to nearest {target.lower()} cell')
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
plt.tight_layout(); finish('5-08_distances', 'Distances to vessels and tumour cells')
pd.Series(dist_summary).unstack()

---
# Part 6: Save results
* `<sample>_adata_annotated.h5ad`: annotated single-cell data, the input for SPACEc neighbourhood analysis.
* `<sample>_cells_annotated.csv`: one row per cell with coordinates, marker z-scores, cluster and cell type.
* `figures/<PLOT_TAG>_sample_summary.csv`: **one row per sample** with all cell-type percentages, key numbers and median distances. Concatenate these files across samples to compare them.
* `<sample>_parameters.json`: every setting used, for the Methods section and for reproducibility.

In [ ]:
adata.write(RES + 'adata_annotated.h5ad')
cells_tab = adata.to_df().join(adata.obs)
cells_tab.to_csv(RES + 'cells_annotated.csv', index=False)
print('written', RES + 'adata_annotated.h5ad', 'and', RES + 'cells_annotated.csv', cells_tab.shape)

# one-row summary per sample (for comparing samples later)
row = {'sample': SAMPLE, 'condition': CONDITION, 'plot_tag': PLOT_TAG, 'roi': ROI}
row.update({f'% {t}': round(v, 2) for t, v in pct.items()})
row.update({f'% {c} (compartment)': round(v, 2) for c, v in comp_pct.items()})
row.update(key_numbers[PLOT_TAG].to_dict())
row.update({f'{q}: {k}': v for (q, k), v in dist_summary.items()})
row.update({f'{t}: {c}': round(v, 1) for t, r in state.iterrows() for c, v in r.items()})
sample_summary = pd.DataFrame([row])
sample_summary.to_csv(FIG_DIR + f'{PLOT_TAG}_sample_summary.csv', index=False)
print('written', FIG_DIR + f'{PLOT_TAG}_sample_summary.csv')

def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o
json.dump(_jsonable(PARAM_LOG), open(RES + 'parameters.json', 'w'), indent=1)
print('written', RES + 'parameters.json')
print(f'\nall figures of this sample: {FIG_DIR}{PLOT_TAG}_*.png')